# UniImage — Python quickstart

`uniimage` is a Cython extension over the UniImage C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install lituus-uniimage
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## Metadata API

In [1]:
import uniimage

uniimage.version(), uniimage.__version__

('1.1.0', '1.1.0')

`read_buffer` parses a still/video container's metadata without decoding pixels. A minimal JPEG (SOI + EOI) carries none:

In [2]:
m = uniimage.read_buffer(bytes([0xFF, 0xD8, 0xFF, 0xD9]))
m.is_valid

False

Strip metadata from a buffer, returning a new bytes object:

In [3]:
out = uniimage.strip_buffer(bytes([0xFF, 0xD8, 0xFF, 0xD9]))
out[:2]

b'\xff\xd8'

## In-place EXIF edit

Open an editable model from a buffer, mutate it, serialize back, then re-read
to confirm the tag round-trips:

In [4]:
jpg = bytes([0xFF, 0xD8, 0xFF, 0xD9])
e = uniimage.edit_buffer(jpg)
e.set_artist("Jane Doe")
e.set_tag("ImageDescription", "quickstart")
edited = e.write()
m = uniimage.read_buffer(edited)
m.get_tag("Artist"), m.get_tag("ImageDescription")

('Jane Doe', 'quickstart')

## Raster API

Decode a two-pixel PPM buffer, resize it, encode it as PNG, and decode the
result again. All encoded and pixel buffers are ordinary Python `bytes`.

In [5]:
ppm = b"P6\n2 1\n255\n" + bytes([255, 0, 0, 0, 0, 255])
image = uniimage.decode_buffer(ppm)
resized = image.resize(4, 2, uniimage.FILTER_BILINEAR)
png = resized.encode(uniimage.FMT_PNG)
roundtrip = uniimage.decode_buffer(png)
(roundtrip.width, roundtrip.height, roundtrip.channels, png[:4])

(4, 2, 3, b'\x89PNG')

Extract a perceptual palette through UniColor's Wu quantizer:

In [6]:
palette = image.extract_palette(2, "wu")
len(palette), palette.color_at(0).space_tag

(2, 15)